# 06: Qwen 3B 균등 문장 결합 QLoRA 학습, Colab

`RUNTIME_PARAMS`의 `train_file`과 `test_file`에 지정한 JSONL 파일을 학습과 검증에 사용합니다.

이 실험의 `min_weight=1.0`은 파트너 가중치를 균등하게 만듭니다. 역빈도 가중치 기능과 실제 학습 조건을 구분하며, 조건 변경은 별도 실험으로 보존합니다.


## 1. 파라미터와 경로


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
RUNTIME_PARAMS = {'prefix_path': '/drive/MyDrive/SiteBoard/06/',
 'model_name': 'Qwen/Qwen2.5-3B-Instruct',
 'train_file': 'train.jsonl',
 'test_file': 'test.jsonl',
 'output_dir': 'models/checkpoints',
 'adapter_path': 'models',
 'run_config_file': 'result/run_config.json',
 'training_history_file': 'result/training_history.json',
 'report_file': 'result/evaluation.json',
 'device': 'cuda',
 'model_revision': None}

HYPER_PARAMS = {
    # 문장 결합 증강 설정, 학습 입력의 조합을 다양화합니다.
    'augmentation': {
        'probability': 0.5,   # 각 학습 샘플에 문장 결합 증강을 적용할 확률입니다.
        'max_concat': 2,      # 한 번에 결합할 원본 샘플 수의 상한입니다.
        'delimiters': [       # 결합 문장 사이에 무작위로 넣을 구분자 후보입니다.
            ' ',
            '  ',
            ' / ',
            ' . ',
            '\n',
            ' , ',
            ' + ',
        ],
        'weighted': True,     # 희소 공종을 우선 선택하는 가중 파트너 샘플링을 켭니다.
        'weight_power': 0.7,  # 공종 역빈도에 적용할 지수로 희소 공종의 선택 비중을 조절합니다.
        'min_weight': 1.0,    # 샘플링 가중치의 하한입니다. 현재 1.0에서는 가중치가 균등해집니다.
    },

    # 학습 기본 설정, 반복 횟수와 최적화 조건을 정합니다.
    'training': {
        'epochs': 3,              # 전체 학습 데이터를 반복해서 학습할 횟수입니다.
        'batch_size': 2,          # 장치당 한 번의 학습 순전파에 넣을 샘플 수입니다.
        'grad_accum': 4,          # 지정한 미니배치 횟수만큼 기울기를 누적한 뒤 가중치를 갱신합니다.
        'learning_rate': 0.0002,  # 최적화기의 기본 학습률로 가중치 갱신 크기를 조절합니다.
        'max_seq_length': 1280,   # 학습과 검증에서 입력과 정답을 합친 토큰 길이의 상한입니다.
        'seed': 42,               # 난수 시드로 학습과 증강의 무작위 동작을 재현하도록 합니다.
    },

    # LoRA 어댑터 설정, 학습할 저차원 가중치의 구조를 정합니다.
    'lora': {
        'r': 16,                   # LoRA 저차원 행렬의 랭크로 학습 용량과 메모리 사용량을 조절합니다.
        'lora_alpha': 32,          # LoRA 갱신량에 적용할 배율의 기준값이며 r과 함께 스케일을 정합니다.
        'lora_dropout': 0.05,      # LoRA 경로의 입력을 일부 제외해 과적합을 줄이는 확률입니다.
        'task_type': 'CAUSAL_LM',  # 인과적 언어 모델 학습에 맞는 PEFT 동작을 선택합니다.
        'bias': 'none',            # 편향 파라미터의 학습 범위이며 none은 편향을 학습하지 않습니다.
        'target_modules': [        # LoRA를 부착할 모델 내부 선형 계층 이름입니다.
            'q_proj',
            'k_proj',
            'v_proj',
            'o_proj',
            'gate_proj',
            'up_proj',
            'down_proj',
        ],
    },

    # 학습 모델의 양자화 설정, 가중치 메모리 사용량을 줄입니다.
    'quantization': {
        'load_in_4bit': True,               # 기본 모델 가중치를 4비트로 불러와 GPU 메모리를 절약합니다.
        'bnb_4bit_quant_type': 'nf4',       # 4비트 가중치의 표현 방식이며 nf4 양자화를 사용합니다.
        'bnb_4bit_use_double_quant': True,  # 양자화 상수도 추가 양자화해 메모리 사용량을 줄입니다.
    },

    # 학습 모델 로딩 설정, 장치 배치와 로딩 자원을 정합니다.
    'model_loading': {
        'device_map': {'': 0},          # 모델 전체를 0번 GPU에 배치합니다. 빈 키는 모델 전체를 뜻합니다.
        'attn_implementation': 'sdpa',  # 어텐션 계산 구현을 선택하며 sdpa는 PyTorch SDPA를 사용합니다.
        'low_cpu_mem_usage': True,      # 모델 로딩 중 CPU 메모리의 최대 사용량을 줄입니다.
    },

    # 학습 토크나이저 설정, 입력 길이를 맞추는 방식을 정합니다.
    'tokenizer': {
        'padding_side': 'right',  # 패딩 토큰을 붙일 방향이며 right는 입력 오른쪽에 붙입니다.
    },

    # 양자화 모델의 학습 준비 설정입니다.
    'kbit_training': {
        'use_gradient_checkpointing': True,  # 역전파 때 중간값을 재계산해 활성값 저장 메모리를 줄입니다.
    },

    # 학습기 설정, 검증과 저장 주기 및 실행 자원을 정합니다.
    'trainer': {
        'per_device_eval_batch_size': 1,  # 학습 중 장치당 검증 배치 크기로 검증 메모리 사용량을 조절합니다.
        'lr_scheduler_type': 'cosine',    # 학습률 변화 방식이며 cosine은 코사인 곡선으로 감소시킵니다.
        'warmup_steps': 10,               # 초기 학습률을 점진적으로 높이는 최적화 스텝 수입니다.
        'logging_steps': 10,              # 학습 로그를 남기는 최적화 스텝 간격입니다.
        'save_strategy': 'epoch',         # 체크포인트 저장 시점이며 epoch는 각 에포크 종료 시 저장합니다.
        'eval_strategy': 'epoch',         # 검증 시점이며 epoch는 각 에포크 종료 시 검증합니다.
        'save_total_limit': 2,            # 유지할 체크포인트 수의 상한으로 저장 공간을 관리합니다.
        'fp16': True,                     # FP16 혼합 정밀도 학습으로 메모리와 연산 비용을 줄입니다.
        'dataloader_num_workers': 2,      # 데이터 로딩에 사용할 작업 프로세스 수입니다.
        'dataloader_pin_memory': True,    # 고정 CPU 메모리로 배치를 준비해 GPU 전송을 돕습니다.
        'report_to': 'none',              # 외부 실험 추적 서비스이며 none은 외부 전송을 끕니다.
        'remove_unused_columns': False,   # 사용하지 않는 데이터 열의 자동 제거 여부를 정합니다.
    },

    # 배치 구성 설정, 샘플 길이와 텐서 형식을 맞춥니다.
    'collator': {
        'pad_to_multiple_of': 8,  # 배치 길이를 지정한 배수로 맞춰 GPU 연산 효율을 돕습니다.
        'return_tensors': 'pt',   # 배치의 반환 형식이며 pt는 PyTorch 텐서를 뜻합니다.
        'padding': True,          # 같은 배치의 샘플 길이를 패딩으로 맞춥니다.
    },

    # 평가 노트북의 추론 설정, 배치 크기와 토큰 상한을 정합니다.
    'evaluation': {
        'batch_size': 2,           # 평가 시 한 번에 추론할 샘플 수로 메모리 사용량을 조절합니다.
        'max_new_tokens': 512,     # 샘플당 새로 생성할 응답 토큰 수의 상한입니다.
        'max_input_tokens': 1024,  # 평가 입력을 토큰화할 때 적용하는 길이 상한입니다.
    },
}


In [ ]:
# Colab GPU 런타임에서 실행하세요. 설치 후 필요하면 런타임을 재시작하세요.
!pip install "torch>=2.2.0" "transformers>=4.46.0,<5" "peft>=0.13.0,<0.18" "accelerate>=0.29.0,<2" "bitsandbytes>=0.43.0,<1" "tqdm>=4.66.0"


## 2. 라이브러리


In [ ]:
import hashlib
import json
import platform
import random
from copy import deepcopy
from pathlib import Path
from typing import Any, Dict, List

import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)


## 3. 학습 결과 JSON 저장


In [ ]:
def runtime_path(field):
    return str(Path(RUNTIME_PARAMS["prefix_path"]) / RUNTIME_PARAMS[field])


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)
        stream.write("\n")


## 4. 학습 및 검증 데이터 읽기


In [ ]:
def read_jsonl(path):
    path = Path(path)
    records = []
    with path.open("r", encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON 문법 오류: {path}:{line_number}") from exc
            messages = record.get("messages") if isinstance(record, dict) else None
            if not isinstance(messages, list) or len(messages) != 3:
                raise ValueError(f"system, user, assistant 메시지 3개가 필요합니다: {path}:{line_number}")
            for message, role in zip(messages, ("system", "user", "assistant")):
                if (not isinstance(message, dict) or message.get("role") != role
                        or not isinstance(message.get("content"), str)):
                    raise ValueError(f"메시지 역할과 문자열 content를 확인하세요: {path}:{line_number}")
            records.append(record)
    if not records:
        raise ValueError(f"데이터가 비어 있습니다: {path}")
    return records


def prepare_training_data():
    train_records = read_jsonl(runtime_path("train_file"))
    valid_records = read_jsonl(runtime_path("test_file"))
    return train_records, valid_records, {
        "train_count": len(train_records),
        "test_count": len(valid_records),
    }


## 5. 실시간 문장 결합 증강 데이터셋


In [ ]:
class QwenChatDataset:
    """
    Qwen ChatML Dataset with On-the-fly Inverse-Frequency Weighted Concatenation.
    - 동일 현장(Same-Site) 제약을 준수하면서, 등장 비율이 낮은 희소 공종(오픈구, 특수마감 등)을
      가중 샘플링(Weighted Sampling)하여 결합 파트너로 우선 선택합니다.
    - Prompt 구간은 -100으로 마스킹하여 Completion(JSON 응답) 구간만 Loss를 계산합니다.
    """

    def __init__(
        self,
        records: List[Dict[str, Any]],
        tokenizer,
        max_seq_length: int,
        is_train: bool,
        aug_prob: float,
        aug_max_concat: int,
        delimiters: List[str],
        use_weighted_sampling: bool,
        weight_power: float,
        min_weight: float
    ):
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.is_train = is_train
        self.aug_prob = aug_prob
        self.aug_max_concat = aug_max_concat
        self.delimiters = delimiters
        self.use_weighted_sampling = use_weighted_sampling and is_train
        self.weight_power = weight_power
        self.min_weight = min_weight

        # 1. 원본 데이터 로드 및 현장(Site)별 그룹화
        self.records: List[Dict[str, Any]] = []
        self.site_to_indices: Dict[str, List[int]] = {}
        self.site_to_weights: Dict[str, List[float]] = {}

        site_wt_counts: Dict[str, Dict[str, int]] = {}

        for idx, data in enumerate(records):
            self.records.append(data)

            site = data.get("metadata", {}).get("site", "COMMON")
            if site not in self.site_to_indices:
                self.site_to_indices[site] = []
                site_wt_counts[site] = {}
            self.site_to_indices[site].append(idx)

            # 공종 빈도 집계
            try:
                y_list = json.loads(data["messages"][2]["content"])
                for item in y_list:
                    wt = item.get("workType", "")
                    site_wt_counts[site][wt] = site_wt_counts[site].get(wt, 0) + 1
            except Exception:
                pass

        # 2. 현장별 희소 공종 역빈도 가중치(Sampling Weights) 사전 계산
        if self.use_weighted_sampling:
            for site, indices in self.site_to_indices.items():
                weights = []
                counts = site_wt_counts.get(site, {})
                for idx in indices:
                    rec = self.records[idx]
                    try:
                        y_list = json.loads(rec["messages"][2]["content"])
                        if y_list:
                            # 해당 샘플의 공종들 중 가장 희소한(빈도가 낮은) 공종의 역빈도 점수 계산
                            rarity_scores = [
                                (1.0 / (counts.get(item.get("workType", ""), 1) ** self.weight_power))
                                for item in y_list
                            ]
                            w = max(rarity_scores)
                        else:
                            w = self.min_weight
                    except Exception:
                        w = self.min_weight

                    weights.append(max(w, self.min_weight))

                # 정규화
                total_w = sum(weights)
                self.site_to_weights[site] = [w / total_w for w in weights] if total_w > 0 else weights

        mode_str = f"학습용 (가중 결합 증강, p={self.aug_prob}, 희소가중={self.use_weighted_sampling})" if is_train else "평가용 (원본 고정)"
        print(f"✓ 데이터셋 로드 완료: {len(self.records)}건 [{mode_str}] (현장 수: {len(self.site_to_indices)})")

    def __len__(self) -> int:
        return len(self.records)

    def _extract_components(self, record: Dict[str, Any]):
        """ChatML 레코드에서 system, rules, x_text, y_list 추출"""
        messages = record["messages"]
        system_prompt = messages[0]["content"]
        user_content = messages[1]["content"]
        assistant_content = messages[2]["content"]

        if "[보드판 텍스트]\n" in user_content:
            rules_part, x_text = user_content.split("[보드판 텍스트]\n", 1)
        else:
            rules_part, x_text = "", user_content

        try:
            y_list = json.loads(assistant_content)
        except Exception:
            y_list = []

        return system_prompt, rules_part, x_text.strip(), y_list

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        base_record = self.records[idx]

        # 실시간 증강 확률 적용 (Train 모드일 때만)
        if self.is_train and random.random() < self.aug_prob:
            site = base_record.get("metadata", {}).get("site", "COMMON")
            same_site_indices = self.site_to_indices.get(site, [])
            same_site_weights = self.site_to_weights.get(site, None) if self.use_weighted_sampling else None

            system_prompt, rules_part, x_base, y_base = self._extract_components(base_record)
            num_concat = random.randint(2, self.aug_max_concat) if self.aug_max_concat >= 2 else 2

            combined_x_list = [x_base]
            combined_y_list = list(y_base)

            for _ in range(num_concat - 1):
                # 희소 공종 가중치(Weighted Sampling) 적용하여 파트너 선택
                if same_site_weights and len(same_site_weights) == len(same_site_indices):
                    partner_idx = random.choices(same_site_indices, weights=same_site_weights, k=1)[0]
                else:
                    partner_idx = random.choice(same_site_indices)

                _, _, x_partner, y_partner = self._extract_components(self.records[partner_idx])
                combined_x_list.append(x_partner)
                combined_y_list.extend(y_partner)

            delimiter = random.choice(self.delimiters)
            new_x_text = delimiter.join(combined_x_list)
            user_content = f"{rules_part}[보드판 텍스트]\n{new_x_text}"
            assistant_content = json.dumps(combined_y_list, ensure_ascii=False, indent=2)
        else:
            messages = base_record["messages"]
            system_prompt = messages[0]["content"]
            user_content = messages[1]["content"]
            assistant_content = messages[2]["content"]

        # 초고속 ChatML 문자열 직접 조립 (Jinja2 오버헤드 제거)
        prompt_text = (
            f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{user_content}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        full_text = f"{prompt_text}{assistant_content}<|im_end|>\n"

        prompt_token_ids = self.tokenizer.encode(prompt_text, add_special_tokens=False)
        full_token_ids = self.tokenizer.encode(
            full_text,
            add_special_tokens=False,
            max_length=self.max_seq_length,
            truncation=True
        )

        prompt_len = min(len(prompt_token_ids), len(full_token_ids))
        labels = [-100] * prompt_len + full_token_ids[prompt_len:]

        # EOS 토큰 보장
        if len(full_token_ids) < self.max_seq_length and full_token_ids[-1] != self.tokenizer.eos_token_id:
            full_token_ids.append(self.tokenizer.eos_token_id)
            labels.append(self.tokenizer.eos_token_id)

        return {
            "input_ids": full_token_ids,
            "attention_mask": [1] * len(full_token_ids),
            "labels": labels
        }


def build_dataset(records, tokenizer, is_train):
    settings = HYPER_PARAMS["augmentation"]
    return QwenChatDataset(
        records=records, tokenizer=tokenizer, max_seq_length=HYPER_PARAMS["training"]["max_seq_length"],
        is_train=is_train, aug_prob=settings["probability"],
        aug_max_concat=settings["max_concat"], delimiters=settings["delimiters"],
        use_weighted_sampling=settings["weighted"], weight_power=settings["weight_power"],
        min_weight=settings["min_weight"],
    )


## 6. 모델, 학습기 구성


In [ ]:
def load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(RUNTIME_PARAMS["model_name"], revision=RUNTIME_PARAMS.get("model_revision"), **HYPER_PARAMS["tokenizer"])
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    return tokenizer


def load_qlora_model():
    settings = HYPER_PARAMS["model_loading"]

    # 양자화 옵션을 적용해 가중치 로드
    model = AutoModelForCausalLM.from_pretrained(
        RUNTIME_PARAMS["model_name"],
        revision=RUNTIME_PARAMS.get("model_revision"),
        torch_dtype=torch.float16,
        quantization_config=BitsAndBytesConfig(
            bnb_4bit_compute_dtype=torch.float16, 
            **HYPER_PARAMS["quantization"],
        ), 
        **settings,
    )

    # 모델 학습에서는 기본적으로 KV 캐시를 사용하지 않음
    model.config.use_cache = False
    
    return model


def add_lora_adapter(model):
    # PEFT 모델로 재구성
    model = get_peft_model(model, LoraConfig(**HYPER_PARAMS["lora"]))
    model.print_trainable_parameters()

    return model


def build_trainer(model, tokenizer, train_records, valid_records):
    settings = HYPER_PARAMS["training"]
    train_dataset = build_dataset(train_records, tokenizer, is_train=True)
    valid_dataset = build_dataset(valid_records, tokenizer, is_train=False)
    
    args = TrainingArguments(
        output_dir=runtime_path("output_dir"), 
        num_train_epochs=settings["epochs"],
        per_device_train_batch_size=settings["batch_size"],
        gradient_accumulation_steps=settings["grad_accum"], 
        learning_rate=settings["learning_rate"],
        seed=settings["seed"], 
        **HYPER_PARAMS["trainer"],
    )
    
    return Trainer(
        model=model, 
        args=args, 
        train_dataset=train_dataset, 
        eval_dataset=valid_dataset,
        data_collator=DataCollatorForSeq2Seq(
            tokenizer=tokenizer, 
            **HYPER_PARAMS["collator"]
        ),
    )


## 7. 보존 경로와 실행 정보


In [ ]:
def collect_execution_metadata(model=None):
    from importlib.metadata import PackageNotFoundError, version

    packages = {}
    for name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes"):
        try:
            packages[name] = version(name)
        except PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(),
        "packages": packages,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "cuda": torch.version.cuda,
        "base_model": RUNTIME_PARAMS["model_name"],
        "base_model_revision": getattr(getattr(model, "config", None), "_commit_hash", None)
            or RUNTIME_PARAMS.get("model_revision"),
    }


def run_training():
    if not torch.cuda.is_available():
        raise RuntimeError("Colab에서 CUDA GPU 런타임을 선택하세요.")
    # 실험의 보존된 데이터와 학습 결과를 덮어쓰지 않습니다.
    for key in ("run_config_file", "training_history_file", "report_file"):
        if Path(runtime_path(key)).exists():
            raise FileExistsError(f"기존 산출물이 있습니다. 새 실험 ID를 사용하세요: {runtime_path(key)}")
    adapter = Path(runtime_path("adapter_path"))
    if adapter.exists() and any(p.name != "README.md" for p in adapter.iterdir()):
        raise FileExistsError(f"기존 모델이 있습니다: {adapter}")
    # 시드 설정
    set_seed(HYPER_PARAMS["training"]["seed"])

    # 지정한 JSONL 파일에서 학습과 검증 데이터를 읽습니다.
    train_records, valid_records, prepared_data = prepare_training_data()
    print(f"학습 {len(train_records)}건 / 검증 {len(valid_records)}건")
    print("GPU:", torch.cuda.get_device_name(0))
    tokenizer = load_tokenizer()

    # 모델 로드 및 어댑터 부착
    model = load_qlora_model()
    model = prepare_model_for_kbit_training(model, **HYPER_PARAMS["kbit_training"])
    model = add_lora_adapter(model)

    # Trainer 인스턴스 생성
    trainer = build_trainer(model, tokenizer, train_records, valid_records)

    # 실행 설정 기록
    prepared_hashes = {
        runtime_path(key): hashlib.sha256(Path(runtime_path(key)).read_bytes()).hexdigest()
        for key in ("train_file", "test_file")
    }
    run_config = {"runtime_params": deepcopy(RUNTIME_PARAMS), "hyper_params": deepcopy(HYPER_PARAMS), "prepared_data": prepared_data, "prepared_file_hashes": prepared_hashes,
                  "execution": collect_execution_metadata(model), "status": "training_started"}
    write_json(runtime_path("run_config_file"), run_config)

    # 학습 실행
    trainer.train()

    # 가중치 저장
    destination = Path(runtime_path("adapter_path"))
    model.save_pretrained(destination)
    tokenizer.save_pretrained(destination)
    write_json(runtime_path("training_history_file"), trainer.state.log_history)
    run_config["status"] = "training_completed"
    write_json(runtime_path("run_config_file"), run_config)
    return str(destination)


## 8. 설정 확인 후 학습

상단의 `RUNTIME_PARAMS`와 `HYPER_PARAMS` 딕셔너리를 먼저 확인하세요. 기본 `min_weight=1.0`을 유지하므로 기존 구현에서 희소 공종 가중치가 균등해질 수 있습니다. 학습 결과가 있는 실험은 새 ID로 복제해 실행하세요.


In [ ]:
print("RUNTIME_PARAMS:", json.dumps(RUNTIME_PARAMS, ensure_ascii=False, indent=2))
print("HYPER_PARAMS:", json.dumps(HYPER_PARAMS, ensure_ascii=False, indent=2))


In [ ]:
adapter_path = run_training()
print("어댑터·토크나이저 저장:", adapter_path)
